# Per-region Benth OU + XGBoost Calibration (Matched Input Design)

This notebook keeps the **same input information** as the FFNN and Wavelet-NN notebooks:

- same CSV input
- same seasonal mean estimation and residualization
- same standardized residual process
- same 30-day lag window
- same current day-of-year harmonic features
- same train / validation / test split

Because gradient-boosted trees do not naturally optimize the OU likelihood jointly over \(\kappa_t\) and \(\sigma_t\), this notebook first forms **local OU pseudo-targets** from the residual series using rolling exact-OU estimates, and then uses **XGBoost** to learn the mapping from the same matched inputs to those time-varying OU parameters.


In [ ]:
# --- Imports ---
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor

np.random.seed(42)


## 1) Load your CSV


In [ ]:
CSV_PATH = "../EDA/region_avg.csv"  # <- change if needed
DATE_COL = "date"
REGION_COL = "region_code"
TEMP_COL = "daily_avg_temperature"

df_raw = pd.read_csv(CSV_PATH)
df_raw = df_raw.rename(columns={DATE_COL: "date", REGION_COL: "region_code", TEMP_COL: "daily_avg_temperature"})

df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw["temp"] = pd.to_numeric(df_raw["daily_avg_temperature"], errors="coerce")
df_raw["region_code"] = pd.to_numeric(df_raw["region_code"], errors="coerce")

df_raw = df_raw.dropna(subset=["date", "temp", "region_code"]).copy()
df_raw["region_code"] = df_raw["region_code"].astype(int)

print(df_raw.head())
print(df_raw.tail())
print("shape:", df_raw.shape)
print("regions:", df_raw["region_code"].nunique())


## 2) Helper functions and local OU pseudo-targets


In [ ]:
def design_matrix_seasonality(dates: pd.Series, K: int = 3, include_trend: bool = True) -> np.ndarray:
    n = len(dates)
    doy = dates.dt.dayofyear.values.astype(float)
    parts = [np.ones(n)]
    if include_trend:
        t = np.arange(n) / 365.0
        parts.append(t)
    for k in range(1, K + 1):
        parts.append(np.sin(2 * np.pi * k * doy / 365.0))
        parts.append(np.cos(2 * np.pi * k * doy / 365.0))
    return np.column_stack(parts)


def fit_seasonal_mean(dates: pd.Series, temp: np.ndarray, K: int = 3):
    X = design_matrix_seasonality(dates, K=K, include_trend=True)
    y = temp.astype(float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    s_hat = X @ beta
    return s_hat, beta


def doy_features(dates: pd.Series, harmonics: int = 3) -> np.ndarray:
    doy = dates.dt.dayofyear.values.astype(float)
    feats = []
    for k in range(1, harmonics + 1):
        feats.append(np.sin(2 * np.pi * k * doy / 365.0))
        feats.append(np.cos(2 * np.pi * k * doy / 365.0))
    return np.column_stack(feats).astype(np.float32)


def make_windows(x: np.ndarray, doy_feat: np.ndarray, window: int = 30):
    x = np.asarray(x, dtype=np.float32)
    doy_feat = np.asarray(doy_feat, dtype=np.float32)
    N, F = doy_feat.shape
    xs, xt, y, idx = [], [], [], []
    for t in range(window, N - 1):
        inp = np.concatenate([x[t - window:t], doy_feat[t]], axis=0)
        xs.append(inp)
        xt.append(x[t])
        y.append(x[t + 1])
        idx.append(t)
    return np.stack(xs), np.array(xt), np.array(y), np.array(idx)


def local_ou_targets(xz: np.ndarray, idx_t: np.ndarray, est_window: int = 30, dt: float = 1.0):
    kappa_list, sigma_list = [], []
    eps = 1e-6
    for t in idx_t:
        start = max(0, int(t) - est_window)
        x_hist = xz[start:int(t)]
        y_hist = xz[start + 1:int(t) + 1]
        if len(x_hist) < 5:
            phi = 0.95
            resid = y_hist - phi * x_hist if len(x_hist) else np.array([0.1], dtype=float)
        else:
            denom = float(np.sum(x_hist ** 2)) + 1e-8
            phi = float(np.sum(x_hist * y_hist) / denom)
            phi = float(np.clip(phi, 0.50, 0.999))
            resid = y_hist - phi * x_hist
        kappa = float(np.clip(-np.log(phi) / dt, 1e-6, 0.5))
        var_resid = float(np.var(resid, ddof=1)) if len(resid) > 1 else float(np.var(resid))
        sigma = float(np.sqrt(max(var_resid * 2.0 * kappa / max(1.0 - np.exp(-2.0 * kappa * dt), 1e-6), 1e-6)))
        kappa_list.append(kappa)
        sigma_list.append(sigma)
    return np.array(kappa_list, dtype=np.float32), np.array(sigma_list, dtype=np.float32)


def compute_skewness(x):
    x = np.asarray(x)
    m = np.mean(x)
    s = np.std(x) + 1e-12
    return np.mean(((x - m) / s) ** 3)


def compute_kurtosis(x):
    x = np.asarray(x)
    m = np.mean(x)
    s = np.std(x) + 1e-12
    return np.mean(((x - m) / s) ** 4)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) + 1e-12
    r2 = 1.0 - ss_res / ss_tot
    corr = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan
    return {"mae": float(mae), "rmse": float(rmse), "r2": float(r2), "corr": float(corr)}


def interval_metrics(y_true, mean_pred, sigma_pred, alpha=0.10):
    zcrit = 1.6448536269514722
    lower = mean_pred - zcrit * sigma_pred
    upper = mean_pred + zcrit * sigma_pred
    covered = ((y_true >= lower) & (y_true <= upper)).astype(float)
    picp = np.mean(covered)
    mpiw = np.mean(upper - lower)
    return {"picp_90": float(picp), "mpiw_90": float(mpiw)}


In [ ]:
# Model / data hyperparameters
K_HARMONICS = 3
DOY_HARMONICS = 3
WINDOW = 30
OU_TARGET_WINDOW = 30
DT = 1.0
KAPPA_MAX = 0.5

regions = sorted(df_raw["region_code"].dropna().astype(int).unique().tolist())
GRID_REGIONS = regions
print("All regions:", regions)


## 3) Per-region training configuration

The structure follows the previous notebooks, but the ML block is now **two XGBoost regressors**:
- one for \(\kappa_t\)
- one for \(\log \sigma_t\)


In [ ]:
# ============================================================
# GRID SEARCH FOR XGBOOST ARCHITECTURE
# ============================================================

GRID_N_ESTIMATORS = [200, 400]
GRID_MAX_DEPTH = [3, 5]
GRID_LEARNING_RATE = [0.03, 0.05]
GRID_SUBSAMPLE = [0.8]
GRID_COLSAMPLE = [0.8]

train_end_date = pd.Timestamp("2013-12-31")
val_end_date = pd.Timestamp("2014-12-31")
test_end_date = pd.Timestamp("2024-12-31")


In [ ]:
# ============================================================
# BUILD BASE REGION ARTIFACTS + DATA PREP
# ============================================================

region_artifacts = {}

for r in regions:
    dfr = (
        df_raw.loc[df_raw["region_code"].astype(int) == int(r), ["date", "region_code", "temp"]]
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )
    if dfr.empty:
        continue

    s_hat, beta = fit_seasonal_mean(dfr["date"], dfr["temp"].to_numpy(dtype=float), K=K_HARMONICS)
    x = dfr["temp"].to_numpy(dtype=float) - s_hat

    region_artifacts[int(r)] = {
        "df": dfr,
        "beta": beta,
        "seasonal_mean": s_hat.astype(np.float32),
        "x": x.astype(np.float32),
    }

GRID_REGIONS = sorted(set(int(r) for r in GRID_REGIONS) & set(region_artifacts.keys()))
print("Final GRID_REGIONS:", GRID_REGIONS)


def prepare_region_data_for_grid(region_id: int):
    art = region_artifacts[int(region_id)]
    dfr = art["df"]
    x = np.asarray(art["x"], dtype=np.float32)

    x_mean = float(np.mean(x))
    x_std = float(np.std(x) + 1e-8)
    xz = ((x - x_mean) / x_std).astype(np.float32)

    doy_feat = doy_features(dfr["date"], harmonics=DOY_HARMONICS)
    X_in, x_t, x_tp1, idx_t = make_windows(xz, doy_feat, window=WINDOW)
    kappa_target, sigma_target = local_ou_targets(xz, idx_t, est_window=OU_TARGET_WINDOW, dt=DT)

    window_dates = dfr.loc[idx_t, "date"].reset_index(drop=True)
    train_mask = window_dates <= train_end_date
    val_mask = (window_dates > train_end_date) & (window_dates <= val_end_date)
    test_mask = (window_dates > val_end_date) & (window_dates <= test_end_date)

    return {
        "dfr": dfr,
        "beta": art["beta"],
        "x_mean": x_mean,
        "x_std": x_std,
        "X_in": X_in.astype(np.float32),
        "x_t": x_t.astype(np.float32),
        "x_tp1": x_tp1.astype(np.float32),
        "idx_t": idx_t,
        "kappa_target": kappa_target.astype(np.float32),
        "sigma_target": sigma_target.astype(np.float32),
        "window_dates": window_dates,
        "train_mask": train_mask.to_numpy(),
        "val_mask": val_mask.to_numpy(),
        "test_mask": test_mask.to_numpy(),
    }


In [ ]:
# ============================================================
# TRAIN / EVALUATE ONE GRID CONFIG
# ============================================================

def fit_and_evaluate_config_for_region(region_id: int, n_estimators: int, max_depth: int,
                                       learning_rate: float, subsample: float, colsample_bytree: float):
    data = prepare_region_data_for_grid(region_id)

    X_in = data["X_in"]
    x_t = data["x_t"]
    x_tp1 = data["x_tp1"]
    kappa_target = data["kappa_target"]
    sigma_target = data["sigma_target"]
    train_mask = data["train_mask"]
    val_mask = data["val_mask"]
    x_mean = data["x_mean"]
    x_std = data["x_std"]

    if train_mask.sum() == 0 or val_mask.sum() == 0:
        return {
            "region": int(region_id),
            "n_estimators": int(n_estimators),
            "max_depth": int(max_depth),
            "learning_rate": float(learning_rate),
            "subsample": float(subsample),
            "colsample_bytree": float(colsample_bytree),
            "status": "bad split",
        }

    X_train = X_in[train_mask]
    X_val = X_in[val_mask]

    kappa_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=4,
    )
    sigma_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=4,
    )

    kappa_model.fit(X_train, kappa_target[train_mask])
    sigma_model.fit(X_train, np.log(np.clip(sigma_target[train_mask], 1e-6, None)))

    kappa_val = np.clip(kappa_model.predict(X_val), 1e-6, KAPPA_MAX)
    sigma_val = np.exp(sigma_model.predict(X_val))
    sigma_val = np.clip(sigma_val, 1e-6, None)

    x_t_val = x_t[val_mask]
    x_tp1_val = x_tp1[val_mask]
    mean_pred_z_val = x_t_val * np.exp(-kappa_val * DT)
    var_pred_z_val = np.clip((sigma_val ** 2) * (1.0 - np.exp(-2.0 * kappa_val * DT)) / (2.0 * kappa_val), 1e-6, None)

    y_true = x_tp1_val * x_std + x_mean
    y_pred = mean_pred_z_val * x_std + x_mean
    sigma_pred = np.sqrt(var_pred_z_val) * x_std

    reg = regression_metrics(y_true, y_pred)
    ivl = interval_metrics(y_true, y_pred, sigma_pred)
    z = (y_true - y_pred) / (sigma_pred + 1e-12)

    return {
        "region": int(region_id),
        "n_estimators": int(n_estimators),
        "max_depth": int(max_depth),
        "learning_rate": float(learning_rate),
        "subsample": float(subsample),
        "colsample_bytree": float(colsample_bytree),
        "status": "ok",
        "test_nll": float(np.mean(0.5 * (np.log(var_pred_z_val) + ((x_tp1_val - mean_pred_z_val) ** 2) / var_pred_z_val))),
        **reg,
        **ivl,
        "coverage_error_90": float(ivl["picp_90"] - 0.90),
        "z_mean": float(np.mean(z)),
        "z_std": float(np.std(z)),
        "z_skew": float(compute_skewness(z)),
        "z_kurtosis": float(compute_kurtosis(z)),
    }


In [ ]:
# ============================================================
# RUN GRID SEARCH
# ============================================================

grid_results = []
search_space = [
    (n_estimators, max_depth, learning_rate, subsample, colsample)
    for n_estimators in GRID_N_ESTIMATORS
    for max_depth in GRID_MAX_DEPTH
    for learning_rate in GRID_LEARNING_RATE
    for subsample in GRID_SUBSAMPLE
    for colsample in GRID_COLSAMPLE
]

print("Total configs per region:", len(search_space))
print("Total regions:", len(GRID_REGIONS))
print("Total fits:", len(search_space) * len(GRID_REGIONS))

for r in GRID_REGIONS:
    print(f"\nRunning region {r}...")
    for n_estimators, max_depth, learning_rate, subsample, colsample in search_space:
        out = fit_and_evaluate_config_for_region(
            region_id=r,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample,
        )
        grid_results.append(out)

grid_results_df = pd.DataFrame(grid_results)
grid_results_df.head()


In [ ]:
# ============================================================
# RANK CONFIGS PER REGION
# ============================================================

grid_ok = grid_results_df[grid_results_df["status"] == "ok"].copy()
grid_ok["abs_cov_error"] = np.abs(grid_ok["coverage_error_90"])
grid_ok["abs_zstd_error"] = np.abs(grid_ok["z_std"] - 1.0)
grid_ok["score"] = (
    1.0 * grid_ok["rmse"]
    + 0.5 * grid_ok["abs_cov_error"]
    + 0.5 * grid_ok["abs_zstd_error"]
    + 0.1 * grid_ok["test_nll"]
)

best_config_per_region = (
    grid_ok.sort_values(["region", "score", "rmse", "abs_cov_error", "abs_zstd_error"])
           .groupby("region", as_index=False)
           .first()
)

best_config_per_region.sort_values("region")


In [ ]:
# ============================================================
# OVERALL ARCHITECTURE SUMMARY
# ============================================================

arch_summary = (
    grid_ok.groupby(["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree"], as_index=False)
           .agg(
               mean_test_nll=("test_nll", "mean"),
               mean_rmse=("rmse", "mean"),
               mean_r2=("r2", "mean"),
               mean_corr=("corr", "mean"),
               mean_picp_90=("picp_90", "mean"),
               mean_cov_error_90=("coverage_error_90", "mean"),
               mean_z_std=("z_std", "mean"),
               mean_score=("score", "mean"),
           )
           .sort_values("mean_score")
           .reset_index(drop=True)
)
arch_summary


In [ ]:
# ============================================================
# FINAL TRAINING CONFIG
# ============================================================

USE_GLOBAL_ARCH = False
FINAL_N_ESTIMATORS = 400
FINAL_MAX_DEPTH = 3
FINAL_LEARNING_RATE = 0.05
FINAL_SUBSAMPLE = 0.8
FINAL_COLSAMPLE = 0.8

FINAL_MODEL_DIR = "../Outputs/final_models_xgboost"
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)


In [ ]:
# ============================================================
# FINAL MODEL HELPERS
# ============================================================

def get_final_arch_for_region(region_id: int):
    region_id = int(region_id)
    if USE_GLOBAL_ARCH:
        return {
            "n_estimators": int(FINAL_N_ESTIMATORS),
            "max_depth": int(FINAL_MAX_DEPTH),
            "learning_rate": float(FINAL_LEARNING_RATE),
            "subsample": float(FINAL_SUBSAMPLE),
            "colsample_bytree": float(FINAL_COLSAMPLE),
        }
    row = best_config_per_region.loc[best_config_per_region["region"] == region_id]
    if row.empty:
        raise ValueError(f"No region-specific architecture found for region {region_id}")
    row = row.iloc[0]
    return {
        "n_estimators": int(row["n_estimators"]),
        "max_depth": int(row["max_depth"]),
        "learning_rate": float(row["learning_rate"]),
        "subsample": float(row["subsample"]),
        "colsample_bytree": float(row["colsample_bytree"]),
    }


def train_final_model_for_region(region_id: int):
    data = prepare_region_data_for_grid(region_id)
    arch = get_final_arch_for_region(region_id)

    X_in = data["X_in"]
    x_t = data["x_t"]
    x_tp1 = data["x_tp1"]
    idx_t = data["idx_t"]
    train_mask = data["train_mask"]
    test_mask = data["test_mask"]
    kappa_target = data["kappa_target"]
    sigma_target = data["sigma_target"]
    x_mean = data["x_mean"]
    x_std = data["x_std"]
    dfr = data["dfr"]
    beta = data["beta"]

    X_train = X_in[train_mask]
    X_test = X_in[test_mask]

    kappa_model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=4,
        **arch,
    )
    sigma_model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=4,
        **arch,
    )

    kappa_model.fit(X_train, kappa_target[train_mask])
    sigma_model.fit(X_train, np.log(np.clip(sigma_target[train_mask], 1e-6, None)))

    kappa_all = np.clip(kappa_model.predict(X_in), 1e-6, KAPPA_MAX)
    sigma_all = np.exp(sigma_model.predict(X_in))
    sigma_all = np.clip(sigma_all, 1e-6, None)

    kappa_test = np.clip(kappa_all[test_mask], 1e-6, KAPPA_MAX)
    sigma_test = np.clip(sigma_all[test_mask], 1e-6, None)
    x_t_test = x_t[test_mask]
    x_tp1_test = x_tp1[test_mask]

    mean_pred_z_test = x_t_test * np.exp(-kappa_test * DT)
    var_pred_z_test = np.clip((sigma_test ** 2) * (1.0 - np.exp(-2.0 * kappa_test * DT)) / (2.0 * kappa_test), 1e-6, None)

    y_true = x_tp1_test * x_std + x_mean
    y_pred = mean_pred_z_test * x_std + x_mean
    sigma_pred = np.sqrt(var_pred_z_test) * x_std

    reg = regression_metrics(y_true, y_pred)
    ivl = interval_metrics(y_true, y_pred, sigma_pred)
    z_test = (y_true - y_pred) / (sigma_pred + 1e-12)

    metrics = {
        "test_nll": float(np.mean(0.5 * (np.log(var_pred_z_test) + ((x_tp1_test - mean_pred_z_test) ** 2) / var_pred_z_test))),
        **reg,
        **ivl,
        "coverage_error_90": float(ivl["picp_90"] - 0.90),
        "z_mean": float(np.mean(z_test)),
        "z_std": float(np.std(z_test)),
        "z_skew": float(compute_skewness(z_test)),
        "z_kurtosis": float(compute_kurtosis(z_test)),
    }

    save_path_kappa = os.path.join(FINAL_MODEL_DIR, f"region_{int(region_id)}_OU_XGB_kappa.json")
    save_path_sigma = os.path.join(FINAL_MODEL_DIR, f"region_{int(region_id)}_OU_XGB_sigma.json")
    kappa_model.save_model(save_path_kappa)
    sigma_model.save_model(save_path_sigma)

    artifact = {
        "df": dfr,
        "beta": beta,
        "arch": arch,
        "x_mean": x_mean,
        "x_std": x_std,
        "idx_t": idx_t,
        "train_mask": train_mask,
        "test_mask": test_mask,
        "kappa_all": kappa_all,
        "sigma_all": sigma_all,
        "z_test": z_test,
        "metrics": metrics,
        "save_path_kappa": save_path_kappa,
        "save_path_sigma": save_path_sigma,
    }
    return (kappa_model, sigma_model), artifact


In [ ]:
# ============================================================
# TRAIN FINAL MODEL PER REGION
# ============================================================

final_models = {}
final_region_artifacts = {}
final_summary_rows = []

for r in GRID_REGIONS:
    print(f"Training final model for region {r}...")
    model_r, art_r = train_final_model_for_region(r)
    final_models[int(r)] = model_r
    final_region_artifacts[int(r)] = art_r
    final_summary_rows.append({
        "region": int(r),
        **art_r["arch"],
        **art_r["metrics"],
    })

final_summary = pd.DataFrame(final_summary_rows).sort_values("region").reset_index(drop=True)
final_summary


In [ ]:
# ============================================================
# FINAL MODEL DIAGNOSTIC PLOTS FOR ONE REGION
# ============================================================

REGION_TO_PLOT = GRID_REGIONS[0]
art = final_region_artifacts[REGION_TO_PLOT]
dfr = art["df"]
dates_k = dfr.loc[art["idx_t"], "date"].values

plt.figure(figsize=(8, 4))
plt.plot(dates_k, art["kappa_all"])
plt.title(f"Region {REGION_TO_PLOT}: predicted kappa_t")
plt.xlabel("date")
plt.ylabel("kappa_t")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(dates_k, art["sigma_all"])
plt.title(f"Region {REGION_TO_PLOT}: predicted sigma_t")
plt.xlabel("date")
plt.ylabel("sigma_t")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(art["z_test"], bins=30)
plt.title(f"Region {REGION_TO_PLOT}: standardized residuals (test)")
plt.xlabel("z")
plt.ylabel("count")
plt.show()


In [ ]:
# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

final_summary[[
    "region", "n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree",
    "test_nll", "mae", "rmse", "r2", "corr",
    "picp_90", "coverage_error_90", "z_mean", "z_std", "z_skew", "z_kurtosis"
]]


In [ ]:
# ============================================================
# EXPORT OU-XGBOOST TEMPERATURE MODELS TO PKL
# ============================================================

CALIB_END = "2014-12-31"
OUT_DIR = Path("../Outputs/OUXGBoostParametersRegionSpecificModels/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "XGBoost-OU Temperature Model (Region-Specific Architecture)"


def export_ou_xgb_models_to_pkls(regions_to_export, final_models, final_region_artifacts,
                                 calib_end=CALIB_END, out_dir=OUT_DIR, save_combined=True):
    region_models = {}
    for reg in regions_to_export:
        reg = int(reg)
        (kappa_model, sigma_model) = final_models[reg]
        art = final_region_artifacts[reg]
        payload = {
            "model_name": MODEL_NAME,
            "region": reg,
            "calibration_end": calib_end,
            "beta": np.asarray(art["beta"]),
            "x_mean": float(art["x_mean"]),
            "x_std": float(art["x_std"]),
            "arch": art["arch"],
            "metrics": art["metrics"],
            "kappa_model_json": Path(art["save_path_kappa"]).name,
            "sigma_model_json": Path(art["save_path_sigma"]).name,
        }
        region_models[reg] = payload
        with open(out_dir / f"region_{reg}_ou_xgb_model.pkl", "wb") as f:
            pickle.dump(payload, f)
    if save_combined:
        with open(out_dir / "all_regions_ou_xgb_models.pkl", "wb") as f:
            pickle.dump(region_models, f)
    return region_models


exported_models = export_ou_xgb_models_to_pkls(GRID_REGIONS, final_models, final_region_artifacts)
print(f"Saved {len(exported_models)} region model files to {OUT_DIR}")
